# SwiftLog Week 5: Machine Learning Engineering

**Mission:** Intelligent Logistics Prediction & Supply Chain Optimization.

This notebook reviews source data, engineered features, model comparisons, evaluation artefacts, and limitations. The full model-training pipeline is `../run_swiftlog_project.py`. Run that script from the project root to regenerate the saved models and outputs before rerunning this notebook.

The supplied data is used as provided for this educational prototype. Results are reported as measured; weak performance is not hidden or described as production-ready.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
DATA = ROOT / 'Data'
PROCESSED = DATA / 'Processed_Datasets'
MODELS = ROOT / 'Models'


## 2. Source data and quality checks
The source tables cover customers, orders, order items, products, shipments, delivery performance, returns, inventory, warehouses, suppliers, and employees. The original file-level data dictionary is brief; calculated model features are documented in `Feature_Data_Dictionary.csv`.

In [ ]:
quality = pd.read_csv(PROCESSED / 'data_quality_summary.csv')
display(quality)

## 3. Inspect the original target distributions

In [ ]:
shipments = pd.read_csv(DATA / 'Original_Datasets' / 'SwiftLog_Shipments.csv')
returns = pd.read_csv(DATA / 'Original_Datasets' / 'SwiftLog_Returns.csv')
orders = pd.read_csv(DATA / 'Original_Datasets' / 'SwiftLog_Orders.csv')
print('Shipments:', len(shipments))
print('Delayed shipments (DelayDays > 0):', int((pd.to_numeric(shipments['DelayDays'], errors='coerce') > 0).sum()))
print('Orders associated with recorded returns:', returns['OrderID'].nunique())
print('Orders:', orders['OrderID'].nunique())

## 4. Engineered feature samples
The delivery-delay target is `DelayDays > 0`. The return target is whether an order appears in the Returns table. Demand features use weekly product quantities and lagged values from prior weeks. Customer value is framed as next-month revenue, using the preceding three months of activity. For the demand and customer-value tasks, evaluation is chronological rather than a random row split.

### Delivery delay features

In [ ]:
features = pd.read_csv(PROCESSED / 'delivery_delay_features.csv')
print('Shape:', features.shape)
display(features.head())

### Product return features

In [ ]:
features = pd.read_csv(PROCESSED / 'product_return_features.csv')
print('Shape:', features.shape)
display(features.head())

### Weekly demand forecasting features

In [ ]:
features = pd.read_csv(PROCESSED / 'demand_forecasting_features.csv')
print('Shape:', features.shape)
display(features.head())

### Customer next-month value features

In [ ]:
features = pd.read_csv(PROCESSED / 'customer_value_features.csv')
print('Shape:', features.shape)
display(features.head())

## 5. Compare the candidate algorithms
The table contains three algorithms per task. `Selected` marks the model stored in `Best_Model.pkl`; `Tuned` indicates that a tuning search was accepted. The selected model metrics are updated in the comparison table to match the saved model.

In [ ]:
comparison = pd.read_excel(MODELS / 'Model_Comparison.xlsx')
display(comparison)

## 6. Review selected models and held-out metrics

In [ ]:
selected = comparison[comparison['Selected'] == True]
display(selected)

## 7. Evaluation visuals

In [ ]:
from IPython.display import Image, display
for name in ['delivery_delay_classification_confusion_matrix.png', 'product_return_classification_confusion_matrix.png']:
    path = MODELS / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
importance = MODELS / 'Feature_Importance.png'
if importance.exists():
    display(Image(filename=str(importance)))

## 8. Interpretation and limitations

- Delivery delay and return classification ROC-AUC values near 0.5 indicate weak class discrimination, even when F1 or accuracy appears high. This is especially important for the imbalanced delay target.
- Negative held-out R-squared for demand or next-month customer value means the selected model performed worse than a simple mean-prediction baseline on the test period.
- The dataset spans a limited period and has limited operational context. Forecast quality may improve with richer timestamps, planned delivery dates, inventory snapshots, and more history.
- Model outputs should support human review only. They are not validated for live logistics decisions.

## 9. Reproduce the outputs

From the project root, run `python run_swiftlog_project.py`. The script preprocesses the source data, builds the features, compares three algorithms for each task, evaluates candidates, saves model bundles and visualizations, and regenerates reports and the presentation. Then rerun this notebook to inspect the refreshed outputs.